In [ ]:
import pathlib, json, csv

with open(
    "/scr/BEHAVIOR-1K/asset_pipeline/artifacts/pipeline/collision_average_volumes_2.json",
    "r",
) as f:
    d = json.load(f)
    collision_average_volumes = d["average_volumes"]
    collision_volumes = d["volumes"]

# For now, get categories from CSV file
categories_by_id = {}
avg_category_specs = {}
with open(
    "/scr/BEHAVIOR-1K/asset_pipeline/metadata/category_mapping.csv", newline=""
) as csvfile:
    reader = csv.DictReader(csvfile)
    for i, row in enumerate(reader):
        # TODO: Use something more authoritative than this entry
        # Get the has_system entry to check if the category is a particle system,
        # and if so, skip it entirely.
        has_system = row["has_system"].strip().lower()
        if has_system == "true":
            continue

        cat_id = i  # Temporarily just use row idx. TODO: Cover everything
        category = row["category"].strip()
        categories_by_id[cat_id] = category

        assert (
            category in collision_average_volumes
        ), f"Category {category} not in collision_average_volumes"

        volume = collision_average_volumes[category]
        mass = (
            float(row["mass (auto)"])
            if row["mass (auto)"] and row["mass (auto)"] != "#DIV/0!"
            else None
        )
        assert mass is not None and mass > 0, f"Invalid mass for category {category}"
        density = mass / volume if mass and volume else None

        avg_category_specs[category] = {
            "mass": mass,
            "volume": volume,
            "density": density,
        }

In [ ]:
avg_category_specs

In [ ]:
from bddl.knowledge_base import *

In [ ]:
# Load particle params and delete everything that shows up there.
import csv

synset_has_particle_density = set()
with open("/scr/BEHAVIOR-1K/asset_pipeline/metadata/substance_hyperparams.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        synset = Synset.get(row["synset"].strip())
        if synset is None:
            print(f"Synset {row['synset']} not found")
            continue
        has_density = "particle_density" in json.loads(row["hyperparams"])
        if has_density:
            synset_has_particle_density.add(synset)
        elif "visualSubstance" not in synset.property_names:
            print(f"Synset {synset} does not have particle density")

synset_is_visual = {
    s for s in Synset.all_objects() if "visualSubstance" in s.property_names
}
ignore_synsets = synset_has_particle_density | synset_is_visual
ignore_categories = {c for s in ignore_synsets for c in s.categories}

In [ ]:
densities = {
    Category.get(k): v["density"]
    for k, v in avg_category_specs.items()
    if Category.get(k) not in ignore_categories
}
keys, values = zip(*sorted(densities.items(), key=lambda x: -x[1]))

In [ ]:
# Print the first 100 values
for i in range(100):
    print(f"{keys[i]}: {values[i]}")

In [ ]:
# Print the last 100 values
for i in range(100):
    print(f"{keys[-i - 1]}: {values[-i - 1]}")

In [ ]:
# How many categories are heavier than 2,000 kg/m^3? (e.g. concrete)
heavy_categories = [c for c, d in densities.items() if d > 2000]
print(f"Number of categories heavier than 2,000 kg/m^3: {len(heavy_categories)}")

# How many categories are lighter than 1 kg/m^3? (e.g. air)
light_categories = [c for c, d in densities.items() if d < 1]
print(f"Number of categories lighter than 1 kg/m^3: {len(light_categories)}")

In [ ]:
# Heaviest and lightest objects in the dataset
masses = {}
for cat_name, mdls in collision_volumes.items():
    if cat_name in [
        "floors",
        "ceilings",
        "walls",
        "driveway",
        "lawn",
        "background",
        "roof",
    ]:
        continue
    cat = Category.get(cat_name)
    if cat in ignore_categories:
        continue
    if cat not in densities:
        print(f"Category {cat} not in densities")
        continue
    for mdl, volume in mdls.items():
        mass = densities[cat] * volume
        masses[f"{cat}-{mdl}"] = mass
sorted_masses = sorted(masses.items(), key=lambda x: -x[1])

# Print the heaviest and lightest objects
print("Heaviest objects:")
for i in range(1000):
    print(f"{sorted_masses[i][0]}: {sorted_masses[i][1]}")